In [ ]:
import numpy as np
import pandas as pd
from pathlib import Path

import anndata as ad
import scanpy as sc
from scipy import sparse

from sklearn.decomposition import TruncatedSVD
from sklearn.model_selection import KFold

import torch
import torch.nn as nn

SEED = 6
np.random.seed(SEED)
torch.manual_seed(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
device = torch.device(DEVICE)

EMB_DIM_PERT = 128   # pert gene embedding dim (from SVD)
EMB_DIM_OUT = 128   # output gene embedding dim (from SVD)
RANK_R = 32    # low-rank interaction size

DROPOUT = 0.10
LR = 2e-3 * 0.6 * 0.6
WD = 1e-4 * 1.5
EPOCHS = 400
BATCH_GENES = 16 # minibatch over perturbed genes
EVAL_EVERY = 5
PATIENCE = 10 # early stopping patience in eval steps

# Gate parameters should match metric structure
GATE_A = 0.0
GATE_B = 0.2
EPS = 1e-12

ROOT = Path(".")

MODEL_SEEDS = [90, 70, 80]
ALPHA_GRID = [0.7, 0.75, 0.778, 0.82, 0.86]
GRAD_CLIP = 1.0
H5AD_PATH = ROOT / "data" / "training_cells.h5ad"  # used ONLY for perts not in gene_columns


In [47]:
means_path = ROOT / "data" / "training_data_means.csv"
valmap_path = ROOT / "data" / "pert_ids_val.csv"
sample_sub_path = ROOT / "data" / "sample_submission.csv"

df_means = pd.read_csv(means_path)
df_valmap = pd.read_csv(valmap_path)
df_sub = pd.read_csv(sample_sub_path)

gene_columns = [c for c in df_means.columns if c != "pert_symbol"]

baseline_mask = df_means["pert_symbol"].astype(str) == "non-targeting"
x_base = df_means.loc[baseline_mask, gene_columns].iloc[0].to_numpy(np.float32)

df_train = df_means.loc[~baseline_mask].reset_index(drop=True)
train_genes = df_train["pert_symbol"].astype(str).to_numpy()

X_train_means = df_train[gene_columns].to_numpy(np.float32)
D_train = X_train_means - x_base[None, :] # (80, 5127) delta vs non-targeting

delta_baseline = D_train.mean(axis=0).astype(np.float32)

# pert_id -> gene symbol for leaderboard (first 60)
val_map = dict(zip(df_valmap["pert_id"].astype(str), df_valmap["pert"].astype(str)))

print("Train perts:", len(train_genes), "G:", len(gene_columns))
print("Sample submission rows:", len(df_sub))
print("Val mapping entries:", len(val_map))


Train perts: 80 G: 5127
Sample submission rows: 120
Val mapping entries: 60


In [48]:
AUGMENT_H5AD = True     # turn on/off augmentation
AUG_P = 1.0             # 1.0 = full bootstrap targets, 0.5 = half bootstrap, 0 = off
BOOT_K = 32              # number of bootstrap samples per perturbation
BOOT_M = 256            # cells per bootstrap sample
BOOT_SEED = SEED        # reproducible precompute

# IMPORTANT: to avoid CV leakage, we fit the h5ad->means affine mapping PER FOLD using tr_idx only.
AUG_SLOPE_CLAMP_MIN = 0.2
AUG_SLOPE_CLAMP_MAX = 5.0
AUG_MIN_FIT_PERTS = 8   # if fewer training perts have h5ad cells, we skip augmentation in that fold

EXP_NAME = "baseline_fixed_missing"

# Always fill missing perts from h5ad (DO NOT TURN OFF)
FILL_MISSING_PERTS = True
H5AD_TOPK = 256

# Experiment A: output gene embeddings from h5ad control cells
USE_UOUT_CTRL=True
UOUT_BLEND_ALPHA = 1   # 0 -> pure SVD U_out, 1 -> pure ctrl U_out
UOUT_CTRL_MODE="replace"  # "replace", "concat", or "blend"

# Experiment B: pert embeddings from h5ad control corr for ALL perts (blend with SVD)
USE_ZCTRL_ALL=True
ZCTRL_BETA = 0.5
ZCTRL_TOPK = 512

# Experiment C: GenePT
USE_GENEPT = True
GENEPT_FUSION = "concat"   # "blend" or "concat"
GENEPT_GAMMA = 0.10
GENEPT_WHERE = "z"  # "z", "u", or "both"
GENEPT_DIR = ROOT / "external" / "genept"
GENEPT_WHICH = "gene_protein"  # "gene" or "gene_protein"

# If concat makes training unstable, scale GenePT down
GENEPT_DIM_Z = 64
GENEPT_DIM_U = 64
GENEPT_SCALE_Z = 0.25
GENEPT_SCALE_U = 0.10
GENEPT_MODE = "svd"  # "svd" or "slice"

USE_DOROTHEA = True
DOROTHEA_PATH = ROOT / "external" / "dorothea" / "dorothea.csv"

DORO_WHERE = "z"          # "z", "u", or "both"
DORO_FUSION = "concat"       # "concat" or "blend"
DORO_GAMMA = 0.10            # for blend

DORO_DIM_Z = 64
DORO_DIM_U = 64
DORO_SCALE_Z = 0.8
DORO_SCALE_U = 0.10

DORO_INCLUDE_UNSIGNED = True
DORO_L2NORM = True
DORO_MIN_EDGES = 1000
DORO_SEED = SEED

DORO_Z_USE_ROW_THEN_COL = True
DORO_Z_MIN_OUTDEG = 1
DORO_Z_MIN_INDEG = 1
DORO_Z_DEBUG = True
DORO_MODE = "directed_svd"

DORO_SPLIT_POS_NEG = True         # separate stimulation vs inhibition channels
DORO_INCLUDE_UNSIGNED_CH = False  # optional third channel: all directed edges
DORO_SIGNED_ONLY = False          # if True, drop edges with neither stim nor inhib label
DORO_UNSIGNED_WEIGHT = 1.0        # weight for unsigned channel (only if enabled)

# STRING (ProtT5 sequence + network embeddings)
# Needs in STRING_DIR:
#   9606.protein.info*.txt*
#   9606.protein.sequence.embeddings*.h5
#   9606.protein.network.embeddings*.h5
STRING_DIR = ROOT / "external" / "string"
STRING_SPECIES = "9606"

USE_STRING_SEQ = True
USE_STRING_NET = False
STRING_WHERE = "both"          # "z", "u", or "both"
STRING_FUSION = "concat"       # "blend" or "concat"
STRING_GAMMA = 0.10            # only for blend
STRING_DIM_Z = 64              # only for concat
STRING_DIM_U = 64              # only for concat
STRING_SCALE_Z = 0.25
STRING_SCALE_U = 0.10
STRING_MODE = "svd"            # "svd" or "slice"

_UNIPROT2SYM_CACHE = None
dorothea_z = dorothea_fb_z = dorothea_u = dorothea_fb_u = None

In [ ]:
import pickle
import scipy.sparse as sp
from sklearn.preprocessing import StandardScaler

_H5AD_CACHE = None

def load_genept_embeddings(genept_dir="external/genept", which="gene_protein"):
    genept_dir = Path(genept_dir)

    if which == "gene":
        fname = "GenePT_gene_embedding_ada_text.pickle"
    elif which == "gene_protein":
        fname = "GenePT_gene_protein_embedding_model_3_text.pickle"

    with open(genept_dir / fname, "rb") as f:
        d = pickle.load(f)

    # Normalize keys and values
    out = {}
    for k, v in d.items():
        kk = str(k).upper()
        out[kk] = np.asarray(v, dtype=np.float32)
    return out

def l2norm_rows(X, eps=1e-12):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)

def reduce_gene_embeddings(g2v, genesU, out_dim, mode="svd", seed=6, l2norm=True):
    """
    g2v: dict {SYMBOL_UPPER: vec}
    genesU: iterable of gene identifiers (already uppercase symbols ideally)
    out_dim: desired reduced dimension
    mode: "svd" or "slice"
    returns: (dict {gene: reduced_vec}, fallback_vec)
    """
    avail = [g for g in genesU if g in g2v]
    if len(avail) < 10:
        print("[genept] too few genes available for reduction:", len(avail))
        return None, None

    X = np.stack([np.asarray(g2v[g], dtype=np.float32) for g in avail], axis=0)

    # Slice mode: cheap, no fit, but usually worse than SVD
    if mode == "slice":
        d = min(int(out_dim), int(X.shape[1]))
        Z = X[:, :d].astype(np.float32)
        if d < out_dim:
            Z = np.concatenate([Z, np.zeros((Z.shape[0], out_dim - d), dtype=np.float32)], axis=1)
        if l2norm:
            Z = l2norm_rows(Z)
        fb = Z.mean(axis=0).astype(np.float32)
        return {avail[i]: Z[i] for i in range(len(avail))}, fb

    # SVD mode: standardize then reduce
    Xs = StandardScaler(with_mean=True, with_std=True).fit_transform(X).astype(np.float32)
    n, d = Xs.shape

    # Clamp components: k must be <= min(n-1, d)
    k = min(int(out_dim), int(d), int(n - 1))
    if k < 1:
        print("[genept] cannot reduce: n, d =", n, d)
        return None, None

    svd = TruncatedSVD(n_components=k, random_state=seed)
    Zk = svd.fit_transform(Xs).astype(np.float32)

    if k < out_dim:
        Z = np.concatenate([Zk, np.zeros((n, out_dim - k), dtype=np.float32)], axis=1)
    else:
        Z = Zk

    if l2norm:
        Z = l2norm_rows(Z)

    fb = Z.mean(axis=0).astype(np.float32)
    return {avail[i]: Z[i] for i in range(len(avail))}, fb

def _pick_pert_col(adata):
    for c in ["sgrna_symbol", "pert_symbol", "pert", "perturbation", "gene", "target_gene"]:
        if c in adata.obs.columns:
            return c

def load_h5ad_ctrl_cache(h5ad_path: Path, gene_columns):
    """
    Cache:
      - Xn (CPM10K + log2(1+x)) sparse
      - ctrl mask
      - var map (GENE->idx)
      - out_idx mapping for the 5127 output genes
      - Xout_z (control cells, standardized per gene)
    """
    global _H5AD_CACHE
    if _H5AD_CACHE is not None:
        return _H5AD_CACHE

    adata = ad.read_h5ad(str(h5ad_path))
    pert_col = _pick_pert_col(adata)

    Xc = adata.X
    if not sp.issparse(Xc):
        Xc = sp.csr_matrix(Xc)
    else:
        Xc = Xc.tocsr()

    cell_sum = np.asarray(Xc.sum(axis=1)).ravel().astype(np.float64)
    scale = (10000.0 / np.clip(cell_sum, 1e-12, None)).astype(np.float64)
    Xn = Xc.multiply(scale[:, None]).tocsr()
    Xn.data = np.log1p(Xn.data) / np.log(2.0)

    obs_pert = adata.obs[pert_col].astype(str).to_numpy()
    obs_pertU = np.char.upper(obs_pert.astype("U"))

    ctrl_mask = (obs_pertU == "NON-TARGETING")

    var_names = adata.var_names.astype(str).to_numpy()
    varU = np.char.upper(var_names.astype("U"))
    var = {varU[i]: i for i in range(len(varU))}

    out_idx = np.array([var[str(g).upper()] for g in gene_columns], dtype=np.int64)

    Xn_ctrl = Xn[ctrl_mask]
    Xout = Xn_ctrl[:, out_idx]
    if sp.issparse(Xout):
        Xout = Xout.toarray()
    Xout = Xout.astype(np.float32)

    mu = Xout.mean(axis=0, keepdims=True)
    sd = Xout.std(axis=0, keepdims=True) + 1e-6
    Xout_z = ((Xout - mu) / sd).astype(np.float32)

    _H5AD_CACHE = dict(
        Xn=Xn,
        Xn_ctrl=Xn_ctrl,
        ctrl_mask=ctrl_mask,
        var=var,
        out_idx=out_idx,
        Xout_z=Xout_z,
        obs_pertU=obs_pertU,
    )
    print("[h5ad] cache built:", "n_cells=", Xn.shape[0], "n_genes=", Xn.shape[1], "n_ctrl=", int(ctrl_mask.sum()))
    return _H5AD_CACHE

def build_u_out_from_ctrl(cache, d_out, seed=SEED):
    svd_ctrl = TruncatedSVD(n_components=d_out, random_state=seed)
    svd_ctrl.fit(cache["Xout_z"])
    return svd_ctrl.components_.T.astype(np.float32)  # (G, d_out)

def build_z_from_ctrl_corr(cache, genesU, P_out, topk=256):
    Xout_z = cache["Xout_z"]
    Xn_ctrl = cache["Xn_ctrl"]
    var = cache["var"]

    out = {}
    for gU in genesU:
        j = var.get(gU, None)
        if j is None:
            continue

        xg = Xn_ctrl[:, j]
        if sp.issparse(xg):
            xg = xg.toarray()
        xg = np.asarray(xg).ravel().astype(np.float32)
        xg = (xg - xg.mean()) / (xg.std() + 1e-6)

        corr = (xg[:, None] * Xout_z).mean(axis=0)
        idx = np.argsort(-np.abs(corr))[:topk]
        w = corr[idx].astype(np.float32)

        z = (w[:, None] * P_out[idx]).sum(axis=0)
        z = z / (np.linalg.norm(z) + 1e-12)
        out[gU] = z.astype(np.float32)

    return out

def build_genept_reduced_tables(genept_dict, genes_for_z, genes_for_u, dim_z, dim_u, seed=6,
                               reducer_mode="svd", l2norm=True):
    z_table, z_fb = reduce_gene_embeddings(genept_dict, genes_for_z, dim_z, mode=reducer_mode, seed=seed, l2norm=l2norm)
    u_table, u_fb = reduce_gene_embeddings(genept_dict, genes_for_u, dim_u, mode=reducer_mode, seed=seed, l2norm=l2norm)
    return z_table, z_fb, u_table, u_fb

def concat_genept(base_vec, genept_vec, scale=0.25, l2_after=False):
    if genept_vec is None:
        out = base_vec
    else:
        out = np.concatenate([base_vec, scale * genept_vec], axis=-1)
    if l2_after:
        out = out / (np.linalg.norm(out) + 1e-12)
    return out.astype(np.float32)

def _looks_like_uniprot(x: str) -> bool:
    s = str(x).strip().upper()
    if len(s) < 6 or len(s) > 10:
        return False
    # light heuristic: many UniProt accessions start with O/P/Q or A0A...
    return s[0] in ["O", "P", "Q", "A"]


def _load_string_uniprot_to_symbol_map_from_paths(info_path: Path, ali_path: Path):
    """
    Build UniProt accession -> HGNC symbol using STRING:
      - protein.info: protein_external_id -> preferred_name (symbol)
      - protein.aliases: UniProt accession -> protein_external_id
    """
    # protein.info
    df_info = pd.read_csv(info_path, sep="\t", comment="#", dtype=str)
    cols = list(df_info.columns)
    if cols and cols[0].startswith("#"):
        df_info = df_info.rename(columns={cols[0]: cols[0].lstrip("#")})

    id_col = df_info.columns[0]
    name_col = df_info.columns[1]

    prot = df_info[id_col].astype(str).to_numpy()
    name = df_info[name_col].astype(str).to_numpy()

    # strip taxonomy prefix "9606."
    protU = np.char.upper(np.char.replace(prot.astype("U"), "9606.", ""))
    nameU = np.char.upper(name.astype("U"))
    ensp_to_sym = dict(zip(protU.tolist(), nameU.tolist()))

    # protein.aliases (no header)
    df_ali = pd.read_csv(ali_path, sep="\t", comment="#", dtype=str, header=None)

    prot_id = df_ali.iloc[:, 0].astype(str).to_numpy()
    alias   = df_ali.iloc[:, 1].astype(str).to_numpy()
    src     = df_ali.iloc[:, 2].astype(str).to_numpy()

    prot_idU = np.char.upper(np.char.replace(prot_id.astype("U"), "9606.", ""))
    aliasU   = np.char.upper(alias.astype("U"))
    srcU     = np.char.upper(src.astype("U"))

    keep = np.char.find(srcU, "UNIPROT") >= 0

    m = {}
    for a, p in zip(aliasU[keep], prot_idU[keep]):
        sym = ensp_to_sym.get(p, None)
        if sym is None:
            continue
        if a not in m:
            m[a] = sym
    return m


def _get_uniprot_to_symbol_map():
    """
    Cached UniProt->symbol map.
    Uses explicit STRING file paths:
      external/string/9606.protein.info.v12.0.txt
      external/string/9606.protein.aliases.v12.0.txt
    """
    global _UNIPROT2SYM_CACHE
    if _UNIPROT2SYM_CACHE is not None:
        return _UNIPROT2SYM_CACHE

    string_dir = ROOT / "external" / "string"
    info_path = string_dir / "9606.protein.info.v12.0.txt"
    ali_path  = string_dir / "9606.protein.aliases.v12.0.txt"

    m = _load_string_uniprot_to_symbol_map_from_paths(info_path, ali_path)
    print("[dorothea] built UniProt->symbol map from STRING:", len(m))
    _UNIPROT2SYM_CACHE = m
    return _UNIPROT2SYM_CACHE


def _read_dorothea_df(path: Path) -> pd.DataFrame:
    # auto-detect delimiter; normalize header names; strip BOM
    df = pd.read_csv(path, sep=None, engine="python")
    df.columns = [str(c).replace("\ufeff", "").strip().lower() for c in df.columns]
    return df


def load_dorothea_edges_signed_unsigned(dorothea_csv: Path, genesU_set: set):
    """
    Returns edges inside genesU_set:
      srcK, tgtK: uppercase symbols (mapped from UniProt via STRING if needed)
      stimK: bool
      inhibK: bool
      undK: bool (directed)
    """
    df = _read_dorothea_df(dorothea_csv)

    src = df["source"].astype(str).str.upper().to_numpy()
    tgt = df["target"].astype(str).str.upper().to_numpy()

    # detect UniProt-like IDs and map via STRING if needed
    sample = src[:50].tolist() + tgt[:50].tolist()
    uniprotish = np.mean([_looks_like_uniprot(x) and (x not in genesU_set) for x in sample]) > 0.5
    if uniprotish:
        m = _get_uniprot_to_symbol_map()
        if len(m) > 0:
            src = np.array([m.get(s, s) for s in src], dtype="U")
            tgt = np.array([m.get(t, t) for t in tgt], dtype="U")

    stim_cols  = [c for c in ["consensus_stimulation", "is_stimulation"] if c in df.columns]
    inhib_cols = [c for c in ["consensus_inhibition", "is_inhibition"] if c in df.columns]

    stim = np.zeros(len(df), dtype=bool)
    inhib = np.zeros(len(df), dtype=bool)
    for c in stim_cols:
        stim |= df[c].astype(bool).to_numpy()
    for c in inhib_cols:
        inhib |= df[c].astype(bool).to_numpy()

    if "is_directed" in df.columns:
        und = df["is_directed"].astype(bool).to_numpy()
    else:
        und = np.ones(len(df), dtype=bool)

    keep = np.array([(s in genesU_set) and (t in genesU_set) for s, t in zip(src, tgt)], dtype=bool)

    print("[dorothea] parsed cols:", list(df.columns)[:12])
    print("[dorothea] edges loaded:", len(df), "kept(in-gene-set):", int(keep.sum()))
    if uniprotish:
        print("[dorothea] detected UniProt-like IDs:", True, "| mapping size:", len(_UNIPROT2SYM_CACHE or {}))

    return src[keep], tgt[keep], stim[keep], inhib[keep], und[keep]


def _svd_rowcol_embeddings(A: sp.csr_matrix, k: int, seed: int):
    """
    For sparse A (G x G):
      row_emb = UΣ  via svd.fit_transform(A)
      col_emb = VΣ  via components_.T * singular_values
    Returns (row_emb, col_emb) each (G, k), padded as needed.
    """
    G = A.shape[0]
    k = int(k)
    k_eff = int(min(max(1, k), G - 1))

    if A.nnz < 10 or k_eff < 1:
        return np.zeros((G, k), dtype=np.float32), np.zeros((G, k), dtype=np.float32)

    svd = TruncatedSVD(n_components=k_eff, random_state=seed)
    row_eff = svd.fit_transform(A).astype(np.float32)  # (G, k_eff) = UΣ
    col_eff = (svd.components_.T.astype(np.float32) *
               svd.singular_values_.astype(np.float32)[None, :]).astype(np.float32)  # (G, k_eff) = VΣ

    if k_eff < k:
        pad = np.zeros((G, k - k_eff), dtype=np.float32)
        row = np.concatenate([row_eff, pad], axis=1)
        col = np.concatenate([col_eff, pad], axis=1)
    else:
        row = row_eff[:, :k]
        col = col_eff[:, :k]

    return row, col


def build_dorothea_directed_svd_embeddings(
    dorothea_csv: Path,
    gene_columns,
    genes_needed,
    dim_z: int,
    dim_u: int,
    seed: int = 6,
    split_pos_neg: bool = True,
    include_unsigned_ch: bool = False,
    signed_only: bool = False,
    unsigned_weight: float = 1.0,
    l2norm: bool = True,
    min_edges: int = 1000,
):
    """
    Directed TF->target SVD.

    Z-space:
      - Z_row: row embeddings (source/regulator) = UΣ
      - Z_col: col embeddings (target)          = VΣ
      - Z per pert: row if outdeg>0 else col if indeg>0 else fallback (fb_u)

    U_out-space:
      - Ug: col embeddings in dim_u (targets) for optional U block
    """
    genesU = [str(g).upper() for g in gene_columns]
    genesU_set = set(genesU)
    g2i = {g: i for i, g in enumerate(genesU)}
    G = len(genesU)

    src, tgt, stim, inhib, und = load_dorothea_edges_signed_unsigned(dorothea_csv, genesU_set=genesU_set)
    if len(src) < int(min_edges):
        print("[dorothea] too few usable edges, skipping:", len(src))
        return None, None, None, None

    if signed_only:
        keep_s = stim | inhib
        src = src[keep_s]; tgt = tgt[keep_s]
        stim = stim[keep_s]; inhib = inhib[keep_s]; und = und[keep_s]
        print("[dorothea] signed_only kept edges:", len(src))

    keep_d = und.astype(bool)
    src = src[keep_d]; tgt = tgt[keep_d]
    stim = stim[keep_d]; inhib = inhib[keep_d]

    if len(src) < int(min_edges):
        print("[dorothea] too few directed edges after filtering, skipping:", len(src))
        return None, None, None, None

    ii = np.array([g2i[s] for s in src], dtype=np.int64)
    jj = np.array([g2i[t] for t in tgt], dtype=np.int64)

    outdeg = np.bincount(ii, minlength=G).astype(np.int32)
    indeg  = np.bincount(jj, minlength=G).astype(np.int32)

    def mkA(mask, val=1.0):
        m = mask.astype(bool)
        if int(m.sum()) == 0:
            return sp.csr_matrix((G, G), dtype=np.float32)
        data = np.full(int(m.sum()), float(val), dtype=np.float32)
        return sp.csr_matrix((data, (ii[m], jj[m])), shape=(G, G), dtype=np.float32)

    dim_z = int(dim_z)
    dim_u = int(dim_u)

    # Allocate dims for Z
    if split_pos_neg:
        if include_unsigned_ch:
            k_any_z = max(1, dim_z // 3)
            rem_z = dim_z - k_any_z
            k_pos_z = max(1, rem_z // 2)
            k_neg_z = max(1, rem_z - k_pos_z)
        else:
            k_pos_z = max(1, dim_z // 2)
            k_neg_z = max(1, dim_z - k_pos_z)
            k_any_z = 0
    else:
        k_pos_z = dim_z; k_neg_z = 0; k_any_z = 0

    # Allocate dims for U
    if split_pos_neg:
        if include_unsigned_ch:
            k_any_u = max(1, dim_u // 3)
            rem_u = dim_u - k_any_u
            k_pos_u = max(1, rem_u // 2)
            k_neg_u = max(1, rem_u - k_pos_u)
        else:
            k_pos_u = max(1, dim_u // 2)
            k_neg_u = max(1, dim_u - k_pos_u)
            k_any_u = 0
    else:
        k_pos_u = dim_u; k_neg_u = 0; k_any_u = 0

    if split_pos_neg:
        A_pos = mkA(stim, val=1.0)
        A_neg = mkA(inhib, val=1.0)
        A_any = mkA(np.ones_like(stim, dtype=bool), val=float(unsigned_weight)) if include_unsigned_ch else None

        Zr_pos, Zc_pos = _svd_rowcol_embeddings(A_pos, k_pos_z, seed=seed)
        Zr_neg, Zc_neg = _svd_rowcol_embeddings(A_neg, k_neg_z, seed=seed + 1)

        Z_row = np.concatenate([Zr_pos, Zr_neg], axis=1).astype(np.float32)
        Z_col = np.concatenate([Zc_pos, Zc_neg], axis=1).astype(np.float32)

        if include_unsigned_ch and A_any is not None:
            Zr_any, Zc_any = _svd_rowcol_embeddings(A_any, k_any_z, seed=seed + 2)
            Z_row = np.concatenate([Z_row, Zr_any], axis=1).astype(np.float32)
            Z_col = np.concatenate([Z_col, Zc_any], axis=1).astype(np.float32)

        Z_row = Z_row[:, :dim_z] if Z_row.shape[1] >= dim_z else np.concatenate([Z_row, np.zeros((G, dim_z - Z_row.shape[1]), np.float32)], axis=1)
        Z_col = Z_col[:, :dim_z] if Z_col.shape[1] >= dim_z else np.concatenate([Z_col, np.zeros((G, dim_z - Z_col.shape[1]), np.float32)], axis=1)

        # U space uses COL embeddings
        _, Uc_pos = _svd_rowcol_embeddings(A_pos, k_pos_u, seed=seed + 10)
        _, Uc_neg = _svd_rowcol_embeddings(A_neg, k_neg_u, seed=seed + 11)
        Ug = np.concatenate([Uc_pos, Uc_neg], axis=1).astype(np.float32)

        if include_unsigned_ch and A_any is not None:
            _, Uc_any = _svd_rowcol_embeddings(A_any, k_any_u, seed=seed + 12)
            Ug = np.concatenate([Ug, Uc_any], axis=1).astype(np.float32)

        Ug = Ug[:, :dim_u] if Ug.shape[1] >= dim_u else np.concatenate([Ug, np.zeros((G, dim_u - Ug.shape[1]), np.float32)], axis=1)

    else:
        ws = np.zeros(len(src), dtype=np.float32)
        ws[stim] = 1.0
        ws[inhib] = -1.0
        A = sp.csr_matrix((ws, (ii, jj)), shape=(G, G), dtype=np.float32)
        Z_row, Z_col = _svd_rowcol_embeddings(A, dim_z, seed=seed)
        Ug = Z_col[:, :dim_u] if dim_u <= Z_col.shape[1] else np.concatenate([Z_col, np.zeros((G, dim_u - Z_col.shape[1]), np.float32)], axis=1)

    if l2norm:
        Z_row = l2norm_rows(Z_row)
        Z_col = l2norm_rows(Z_col)
        Ug = l2norm_rows(Ug)

    dorothea_u = {genesU[i]: Ug[i].copy() for i in range(G)}
    fb_u = Ug.mean(axis=0).astype(np.float32)

    # DORO_Z_USE_ROW_THEN_COL, DORO_Z_MIN_OUTDEG, DORO_Z_MIN_INDEG, DORO_Z_DEBUG
    # Z-shaped fallback (IMPORTANT when dim_z != dim_u)
    fbz_tmp = (Z_row.mean(axis=0) + Z_col.mean(axis=0)).astype(np.float32) * 0.5  # (dim_z,)

    dorothea_z = {}
    used_row = used_col = used_fb = 0

    for p in genes_needed:
        pU = str(p).upper()
        if pU not in g2i:
            dorothea_z[pU] = fbz_tmp.copy()
            used_fb += 1
            continue

        idx = g2i[pU]
        if DORO_Z_USE_ROW_THEN_COL and (outdeg[idx] >= int(DORO_Z_MIN_OUTDEG)):
            dorothea_z[pU] = Z_row[idx].copy()
            used_row += 1
        elif DORO_Z_USE_ROW_THEN_COL and (indeg[idx] >= int(DORO_Z_MIN_INDEG)):
            dorothea_z[pU] = Z_col[idx].copy()
            used_col += 1
        else:
            dorothea_z[pU] = fbz_tmp.copy()
            used_fb += 1

    fb_z = np.stack(list(dorothea_z.values()), axis=0).mean(axis=0).astype(np.float32)

    if DORO_Z_DEBUG:
        trainU = [str(g).upper() for g in train_genes.tolist()] if "train_genes" in globals() else []
        if trainU:
            tr_row = sum((g in g2i) and (outdeg[g2i[g]] >= int(DORO_Z_MIN_OUTDEG)) for g in trainU)
            tr_col = sum((g in g2i) and (outdeg[g2i[g]] < int(DORO_Z_MIN_OUTDEG)) and (indeg[g2i[g]] >= int(DORO_Z_MIN_INDEG)) for g in trainU)
            tr_fb  = len(trainU) - tr_row - tr_col
            print(f"[dorothea] Z policy (train perts): row={tr_row} col={tr_col} fb={tr_fb} (n={len(trainU)})")

        print(f"[dorothea] Z policy (genes_needed): row={used_row} col={used_col} fb={used_fb} (n={len(genes_needed)})")
        print("[dorothea] degree stats:",
              "outdeg>0:", int((outdeg > 0).sum()),
              "indeg>0:", int((indeg > 0).sum()),
              "G:", G)

    print("[dorothea] directed_svd built:",
          "dim_z=", dim_z, "dim_u=", dim_u,
          "split_pos_neg=", split_pos_neg,
          "unsigned_ch=", include_unsigned_ch,
          "signed_only=", signed_only)
    return dorothea_z, fb_z, dorothea_u, fb_u

# =============================
# STRING embeddings loader
# =============================
_STRING_ID2SYM_CACHE = None
_STRING_RAW_CACHE = {}  # key=(species, kind) -> (gene2vec_raw, fb_raw)

def _find_first_glob(dirpath: Path, patterns):
    for pat in patterns:
        hits = sorted(dirpath.glob(pat))
        if hits:
            return hits[0]
    return None

def get_string_id_to_symbol_map(string_dir: Path, species: str = "9606"):
    global _STRING_ID2SYM_CACHE
    if _STRING_ID2SYM_CACHE is not None:
        return _STRING_ID2SYM_CACHE

    info_path = _find_first_glob(
        string_dir,
        patterns=[
            f"{species}.protein.info*.txt.gz",
            f"{species}.protein.info*.txt",
        ],
    )

    df = pd.read_csv(info_path, sep="\t", comment="#", dtype=str)
    cols = list(df.columns)
    if cols and cols[0].startswith("#"):
        df = df.rename(columns={cols[0]: cols[0].lstrip("#")})

    id_col = df.columns[0]
    name_col = df.columns[1]

    prot_ids = df[id_col].astype(str).to_numpy()
    names = df[name_col].astype(str).str.upper().to_numpy()

    _STRING_ID2SYM_CACHE = dict(zip(prot_ids, names))
    return _STRING_ID2SYM_CACHE

def load_string_raw_embeddings(string_dir: Path, kind: str, species: str = "9606"):
    """
    kind: 'sequence' or 'network'
    Expects H5 datasets: 'embeddings' and 'proteins'
    """
    key = (species, kind)
    if key in _STRING_RAW_CACHE:
        return _STRING_RAW_CACHE[key]

    import h5py

    h5_path = _find_first_glob(
        string_dir,
        patterns=[
            f"{species}.protein.{kind}.embeddings*.h5",
            f"{species}.protein.{kind}.embeddings*.hdf5",
        ],
    )

    id2sym = get_string_id_to_symbol_map(string_dir, species=species)

    with h5py.File(h5_path, "r") as f:
        E = f["embeddings"][:].astype(np.float32)
        P = f["proteins"][:]

    # proteins often stored as bytes
    prots = []
    for p in P:
        if isinstance(p, (bytes, np.bytes_)):
            prots.append(p.decode("utf-8"))
        else:
            prots.append(str(p))

    # Map STRING protein_id -> symbol, average if duplicates
    sums = {}
    cnts = {}
    for pid, vec in zip(prots, E):
        sym = id2sym.get(pid, None)
        if sym is None:
            continue
        gU = str(sym).upper()
        if gU not in sums:
            sums[gU] = vec.copy()
            cnts[gU] = 1
        else:
            sums[gU] += vec
            cnts[gU] += 1

    gene2vec = {gU: (sums[gU] / float(cnts[gU])).astype(np.float32) for gU in sums.keys()}
    fb = (E.mean(axis=0).astype(np.float32) if E.shape[0] > 0 else None)

    _STRING_RAW_CACHE[key] = (gene2vec, fb)
    return gene2vec, fb

def build_string_reduced_tables(string_dir: Path, genes_list, out_dim: int, kind: str,
                                mode: str = "svd", seed: int = 0, species: str = "9606"):
    raw, fb_raw = load_string_raw_embeddings(string_dir, kind=kind, species=species)
    red, fb = reduce_gene_embeddings(raw, genes_list, out_dim=int(out_dim), mode=mode, seed=int(seed))
    return red, fb

Build gene embeddings from D_train (SVD on signed-log transformed deltas). Build embeddings for output genes (gene_columns) via SVD on (80, 5127). Pert embeddings are looked up by gene symbol in gene_columns, otherwise fallback to the mean embedding.

In [ ]:
val_targets = df_valmap["pert"].astype(str).tolist()

genes_needed = sorted(set([str(g).upper() for g in train_genes.tolist()] +
                          [str(g).upper() for g in val_targets]))

geneU = pd.Index([str(g).upper() for g in gene_columns])

# signed log transform (handles negative deltas)
X = D_train.astype(np.float32, copy=True)
X = np.sign(X) * np.log2(1.0 + np.abs(X))

svd = TruncatedSVD(n_components=max(EMB_DIM_PERT, EMB_DIM_OUT), random_state=SEED)
svd.fit(X)

gene_emb_all = svd.components_.T.astype(np.float32)  # (G, k)
k_svd = gene_emb_all.shape[1]
d_pert = min(int(EMB_DIM_PERT), int(k_svd))
d_out  = min(int(EMB_DIM_OUT),  int(k_svd))

print("SVD k:", k_svd, "gene_emb_all:", gene_emb_all.shape)
print("Effective dims:", "d_pert=", d_pert, "d_out=", d_out)

# base dictionaries (only for genes in gene_columns)
gene2emb_pert_svd = {gene_columns[i].upper(): gene_emb_all[i, :d_pert].copy()
                     for i in range(len(gene_columns))}
gene2emb_out_svd  = {gene_columns[i].upper(): gene_emb_all[i, :d_out ].copy()
                     for i in range(len(gene_columns))}

emb_fallback_pert = gene_emb_all[:, :d_pert].mean(axis=0).astype(np.float32)
emb_fallback_out  = gene_emb_all[:, :d_out ].mean(axis=0).astype(np.float32)

# Missing perts lists (these are the ones that were killing fairness)
missing_train = [g for g in train_genes.tolist() if str(g).upper() not in geneU]
missing_val   = [g for g in val_targets           if str(g).upper() not in geneU]
missing_allU  = sorted(set([str(g).upper() for g in (missing_train + missing_val)]))

if missing_train:
    print(f"train perts not in gene_columns: {len(missing_train)}. Example: {missing_train[:12]}")
if missing_val:
    print(f"val perts not in gene_columns: {len(missing_val)}. Example: {missing_val[:12]}")

# ---------------------------------
# h5ad: build embeddings
# ---------------------------------
missing_pert_h5ad = {}
zctrl_all = {}

U_ctrl = None

if (FILL_MISSING_PERTS and len(missing_allU) > 0) or USE_ZCTRL_ALL or USE_UOUT_CTRL:
    cache = load_h5ad_ctrl_cache(H5AD_PATH, gene_columns)
    P_out = gene_emb_all[:, :d_pert].astype(np.float32)  # (G, d_pert) basis

    # Fill missing perts (critical)
    if FILL_MISSING_PERTS and len(missing_allU) > 0:
        tmp = build_z_from_ctrl_corr(cache, missing_allU, P_out, topk=H5AD_TOPK)
        missing_pert_h5ad.update(tmp)
        print("[h5ad] embedded missing perts:", len(missing_pert_h5ad), "of", len(missing_allU))

    # Optional: build h5ad corr for ALL perts (for blending experiment)
    if USE_ZCTRL_ALL:
        tmp = build_z_from_ctrl_corr(cache, genes_needed, P_out, topk=ZCTRL_TOPK)
        zctrl_all.update(tmp)
        print("[h5ad] embedded ALL perts for blending:", len(zctrl_all), "of", len(genes_needed))

    # Optional: U_out from control cells
    if USE_UOUT_CTRL:
        U_ctrl = build_u_out_from_ctrl(cache, d_out=d_out, seed=SEED)  # (G, d_out)
        print("[h5ad] U_ctrl:", U_ctrl.shape)

# ---------------------------------
# GenePT: load + reduce
# ---------------------------------
genept_red_z = None
genept_fb_z  = None
genept_red_u = None
genept_fb_u  = None

if USE_GENEPT:
    g2v = load_genept_embeddings(GENEPT_DIR, which=GENEPT_WHICH)

    if GENEPT_FUSION == "blend":
        dim_z = int(d_pert)
        dim_u = int(d_out)
    elif GENEPT_FUSION == "concat":
        dim_z = int(GENEPT_DIM_Z)
        dim_u = int(GENEPT_DIM_U)

    if (GENEPT_WHERE in ["z", "both"]):
        genept_red_z, genept_fb_z = reduce_gene_embeddings(
            g2v, genes_needed, out_dim=dim_z, mode=GENEPT_MODE, seed=SEED
        )

    if (GENEPT_WHERE in ["u", "both"]):
        genept_red_u, genept_fb_u = reduce_gene_embeddings(
            g2v, [str(g).upper() for g in gene_columns], out_dim=dim_u, mode=GENEPT_MODE, seed=SEED
        )

    print("[genept] reduced:",
          "z_avail=", (0 if genept_red_z is None else len(genept_red_z)),
          "u_avail=", (0 if genept_red_u is None else len(genept_red_u)),
          "dim_z=", (None if genept_fb_z is None else genept_fb_z.shape[0]),
          "dim_u=", (None if genept_fb_u is None else genept_fb_u.shape[0]))

# ---------------------------------
# DoRothEA directed SVD embeddings
# ---------------------------------
dorothea_z = None
dorothea_fb_z = None
dorothea_u = None
dorothea_fb_u = None

if USE_DOROTHEA:
    if DORO_FUSION == "blend":
        dim_z = int(d_pert)
        dim_u = int(d_out)
    elif DORO_FUSION == "concat":
        dim_z = int(DORO_DIM_Z)
        dim_u = int(DORO_DIM_U)

    if DORO_MODE == "directed_svd":
        dorothea_z, dorothea_fb_z, dorothea_u, dorothea_fb_u = build_dorothea_directed_svd_embeddings(
            DOROTHEA_PATH,
            gene_columns=gene_columns,
            genes_needed=genes_needed,
            dim_z=dim_z,
            dim_u=dim_u,
            seed=DORO_SEED,
            split_pos_neg=bool(DORO_SPLIT_POS_NEG),
            include_unsigned_ch=bool(DORO_INCLUDE_UNSIGNED_CH),
            signed_only=bool(DORO_SIGNED_ONLY),
            unsigned_weight=float(DORO_UNSIGNED_WEIGHT),
            l2norm=bool(DORO_L2NORM),
            min_edges=int(DORO_MIN_EDGES),
        )

    print("[dorothea] tables:",
          "z_avail=", (0 if dorothea_z is None else len(dorothea_z)),
          "u_avail=", (0 if dorothea_u is None else len(dorothea_u)),
          "dim_z=", (None if dorothea_fb_z is None else dorothea_fb_z.shape[0]),
          "dim_u=", (None if dorothea_fb_u is None else dorothea_fb_u.shape[0]))

# ---------------------------------
# STRING embeddings (sequence + network)
# ---------------------------------
string_seq_z = string_seq_fb_z = string_seq_u = string_seq_fb_u = None
string_net_z = string_net_fb_z = string_net_u = string_net_fb_u = None

if USE_STRING_SEQ or USE_STRING_NET:
    if STRING_FUSION == "blend":
        dim_z = int(d_pert)
        dim_u = int(d_out)
    elif STRING_FUSION == "concat":
        dim_z = int(STRING_DIM_Z)
        dim_u = int(STRING_DIM_U)

    if USE_STRING_SEQ:
        if STRING_WHERE in ["z", "both"]:
            string_seq_z, string_seq_fb_z = build_string_reduced_tables(
                STRING_DIR, genes_needed, out_dim=dim_z, kind="sequence",
                mode=STRING_MODE, seed=SEED, species=STRING_SPECIES
            )
        if STRING_WHERE in ["u", "both"]:
            string_seq_u, string_seq_fb_u = build_string_reduced_tables(
                STRING_DIR, [str(g).upper() for g in gene_columns], out_dim=dim_u, kind="sequence",
                mode=STRING_MODE, seed=SEED, species=STRING_SPECIES
            )

    if USE_STRING_NET:
        if STRING_WHERE in ["z", "both"]:
            string_net_z, string_net_fb_z = build_string_reduced_tables(
                STRING_DIR, genes_needed, out_dim=dim_z, kind="network",
                mode=STRING_MODE, seed=SEED, species=STRING_SPECIES
            )
        if STRING_WHERE in ["u", "both"]:
            string_net_u, string_net_fb_u = build_string_reduced_tables(
                STRING_DIR, [str(g).upper() for g in gene_columns], out_dim=dim_u, kind="network",
                mode=STRING_MODE, seed=SEED, species=STRING_SPECIES
            )

    print("[string] reduced:",
          "seq_z=", (0 if string_seq_z is None else len(string_seq_z)),
          "seq_u=", (0 if string_seq_u is None else len(string_seq_u)),
          "net_z=", (0 if string_net_z is None else len(string_net_z)),
          "net_u=", (0 if string_net_u is None else len(string_net_u)),
          "dim_z=", (None if string_seq_fb_z is None else string_seq_fb_z.shape[0]),
          "dim_u=", (None if string_seq_fb_u is None else string_seq_fb_u.shape[0]))



# ============================================================
# COVERAGE FIXES: alias + cross-source projection imputation
#   Goal: avoid "mean-vector" fallbacks for missing genes in
#   GenePT / STRING by predicting missing vectors from DoRothEA
#   (which has 100% coverage in this setup).
# ============================================================

COVERAGE_IMPUTE = True  # set False for ablation

from sklearn.linear_model import Ridge

# Manual symbol aliases (small but high-impact)
# - TAFAZZIN is commonly "TAZ" in many resources
MANUAL_ALIAS = {
    "TAFAZZIN": "TAZ",
}

def _apply_manual_alias_to_dict(d, alias_map=MANUAL_ALIAS):
    if d is None:
        return
    for k, v in alias_map.items():
        kU = str(k).upper()
        vU = str(v).upper()
        if (kU not in d) and (vU in d):
            d[kU] = d[vU]
    # also add reverse if needed (harmless)
    for k, v in list(alias_map.items()):
        kU = str(k).upper()
        vU = str(v).upper()
        if (vU not in d) and (kU in d):
            d[vU] = d[kU]

def _impute_from_anchor(target_dict, anchor_dict, genes_list, ridge_alpha=1.0, name="table"):
    """
    Fill missing genes in target_dict using a ridge map from anchor->target
    trained on overlapping genes.
    """
    if target_dict is None or anchor_dict is None:
        return target_dict

    genesU = [str(g).upper() for g in genes_list]

    overlap = [g for g in genesU if (g in target_dict) and (g in anchor_dict)]
    missing = [g for g in genesU if (g not in target_dict) and (g in anchor_dict)]

    if len(missing) == 0:
        return target_dict

    if len(overlap) < 25:
        print(f"[impute] {name}: overlap too small ({len(overlap)}). Skipping.")
        return target_dict

    X = np.stack([anchor_dict[g] for g in overlap], axis=0).astype(np.float32)
    Y = np.stack([target_dict[g] for g in overlap], axis=0).astype(np.float32)

    # Fit multi-output ridge: Y ≈ X @ W
    rr = Ridge(alpha=float(ridge_alpha), fit_intercept=False)
    rr.fit(X, Y)  # coef_: (d_out, d_in)

    Wt = rr.coef_.T.astype(np.float32)  # (d_in, d_out)

    for g in missing:
        x = anchor_dict[g].astype(np.float32)
        y = (x @ Wt).astype(np.float32)
        # normalize to match conventions (most tables are L2-normalized already)
        y = y / (np.linalg.norm(y) + 1e-12)
        target_dict[g] = y

    print(f"[impute] {name}: filled {len(missing)} missing using anchor map (overlap={len(overlap)})")
    return target_dict

# 1) Alias patching first (helps TAFAZZIN, readthrough quirks sometimes)
_apply_manual_alias_to_dict(genept_red_z)
_apply_manual_alias_to_dict(genept_red_u)
_apply_manual_alias_to_dict(string_seq_z)
_apply_manual_alias_to_dict(string_seq_u)
_apply_manual_alias_to_dict(string_net_z)
_apply_manual_alias_to_dict(string_net_u)

# 2) Cross-source projection fills (DoRothEA is anchor because it has full coverage)
if COVERAGE_IMPUTE and USE_DOROTHEA:
    # Z tables (genes_needed)
    genept_red_z  = _impute_from_anchor(genept_red_z,  dorothea_z, genes_needed, ridge_alpha=1.0, name="genept_red_z <- dorothea_z")
    string_seq_z  = _impute_from_anchor(string_seq_z,  dorothea_z, genes_needed, ridge_alpha=1.0, name="string_seq_z <- dorothea_z")
    string_net_z  = _impute_from_anchor(string_net_z,  dorothea_z, genes_needed, ridge_alpha=1.0, name="string_net_z <- dorothea_z")

    # U tables (gene_columns)
    genept_red_u  = _impute_from_anchor(genept_red_u,  dorothea_u, gene_columns, ridge_alpha=1.0, name="genept_red_u <- dorothea_u")
    string_seq_u  = _impute_from_anchor(string_seq_u,  dorothea_u, gene_columns, ridge_alpha=1.0, name="string_seq_u <- dorothea_u")
    string_net_u  = _impute_from_anchor(string_net_u,  dorothea_u, gene_columns, ridge_alpha=1.0, name="string_net_u <- dorothea_u")
# ============================================================
# PRIORITY 2: ALIGN EXTERNAL EMBEDDINGS INTO SVD SPACE (NO CONCAT)
# - Keep SVD deltas as canonical latent space (d_pert, d_out)
# - Learn ridge maps ext -> svd for Z and U
# - Add aligned priors (same dims), no concat, no dim growth
# Result:
#   Z_train: (N, d_pert)
#   U_out  : (G, d_out)
# ============================================================

from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler

ALIGN_EXTERNAL = True

# weights: start small, then tune
LAM_CTRL_U    = 0.25   # control-cell U prior (mapped into SVD-U space)
LAM_GENEPT_Z  = 0.10
LAM_DORO_Z    = 0.10
LAM_STRSEQ_Z  = 0.10
LAM_STRNET_Z  = 0.10

LAM_GENEPT_U  = 0.10
LAM_DORO_U    = 0.10
LAM_STRSEQ_U  = 0.10
LAM_STRNET_U  = 0.10

ALIGN_RIDGE_ALPHA_Z = 1.0
ALIGN_RIDGE_ALPHA_U = 1.0
ALIGN_MIN_OVERLAP_Z = 12      # perts are only ~80, keep this low
ALIGN_MIN_OVERLAP_U = 256     # genes are 5127, keep this meaningful

def _l2_rows(X, eps=1e-12):
    return X / (np.linalg.norm(X, axis=1, keepdims=True) + eps)

def _fit_ridge_map_dict_to_svd(ext_dict, ext_fb, svd_dict, genes_list, *,
                               alpha: float, min_overlap: int, name: str):
    """
    Fit: svd_vec ≈ Ridge( Standardize(ext_vec) )
    Returns: (W, b, mu, sd) where:
      - ext_std = (ext - mu) / sd
      - pred = ext_std @ W + b
    Shapes:
      W: (d_in, d_out), b: (d_out,), mu/sd: (d_in,)
    """
    if (ext_dict is None) or (svd_dict is None):
        print(f"[align] {name}: missing dict(s), skip")
        return None

    genesU = [str(g).upper() for g in genes_list]
    overlap = [g for g in genesU if (g in ext_dict) and (g in svd_dict)]

    if len(overlap) < int(min_overlap):
        print(f"[align] {name}: overlap too small ({len(overlap)} < {min_overlap}), skip")
        return None

    X = np.stack([ext_dict[g] for g in overlap], axis=0).astype(np.float32)
    Y = np.stack([svd_dict[g] for g in overlap], axis=0).astype(np.float32)

    sc = StandardScaler(with_mean=True, with_std=True)
    Xs = sc.fit_transform(X).astype(np.float32)

    rr = Ridge(alpha=float(alpha), fit_intercept=True)
    rr.fit(Xs, Y)

    # sklearn: coef_ shape (d_out, d_in)
    W = rr.coef_.T.astype(np.float32)          # (d_in, d_out)
    b = rr.intercept_.astype(np.float32)       # (d_out,)
    mu = sc.mean_.astype(np.float32)           # (d_in,)
    sd = sc.scale_.astype(np.float32)          # (d_in,)

    print(f"[align] {name}: fit ok overlap={len(overlap)}  d_in={X.shape[1]} -> d_out={Y.shape[1]}")
    return (W, b, mu, sd)

def _map_vec_dict(gU: str, ext_dict, ext_fb, map_pack):
    """
    Map ext vector for gene into SVD space using fitted (W,b,mu,sd).
    """
    if map_pack is None:
        return None

    W, b, mu, sd = map_pack
    x = None
    if ext_dict is not None:
        x = ext_dict.get(gU, None)
    if x is None:
        x = ext_fb
    if x is None:
        return None

    x = np.asarray(x, dtype=np.float32)
    xs = (x - mu) / (sd + 1e-12)
    y = xs @ W + b
    return y.astype(np.float32)

def _fit_ridge_map_matrix_to_svd(X_ext, Y_svd, *, alpha: float, name: str):
    """
    Same as dict mapper but for matrices:
      X_ext: (G, d_in)
      Y_svd: (G, d_out)
    Returns (W, b, mu, sd) with same conventions.
    """
    X = np.asarray(X_ext, dtype=np.float32)
    Y = np.asarray(Y_svd, dtype=np.float32)

    sc = StandardScaler(with_mean=True, with_std=True)
    Xs = sc.fit_transform(X).astype(np.float32)

    rr = Ridge(alpha=float(alpha), fit_intercept=True)
    rr.fit(Xs, Y)

    W = rr.coef_.T.astype(np.float32)
    b = rr.intercept_.astype(np.float32)
    mu = sc.mean_.astype(np.float32)
    sd = sc.scale_.astype(np.float32)

    print(f"[align] {name}: fit ok  d_in={X.shape[1]} -> d_out={Y.shape[1]}")
    return (W, b, mu, sd)

def _map_rows_matrix(X_ext, map_pack):
    W, b, mu, sd = map_pack
    X = np.asarray(X_ext, dtype=np.float32)
    Xs = (X - mu[None, :]) / (sd[None, :] + 1e-12)
    Y = Xs @ W + b[None, :]
    return Y.astype(np.float32)

# ------------------------------------------------------------
# Base U_out in SVD space (canonical)
# ------------------------------------------------------------
U_out_base = np.vstack([gene2emb_out_svd.get(str(g).upper(), emb_fallback_out)
                        for g in gene_columns]).astype(np.float32)   # (G, d_out)

U_out = U_out_base.copy()

# Optional: add control-cell U prior, but aligned into SVD-U space
# (this replaces "U_out = U_ctrl" which threw away SVD geometry)

if ALIGN_EXTERNAL and USE_UOUT_CTRL and (U_ctrl is not None):
    # Fit map U_ctrl -> U_out_base using all output genes
    map_ctrl_u = _fit_ridge_map_matrix_to_svd(U_ctrl, U_out_base, alpha=ALIGN_RIDGE_ALPHA_U, name="U_ctrl -> U_svd")
    U_ctrl_aligned = _map_rows_matrix(U_ctrl, map_ctrl_u)  # (G, d_out)
    U_out = U_out + float(LAM_CTRL_U) * U_ctrl_aligned
    print(f"[align] U_out += {LAM_CTRL_U:.3f} * aligned(U_ctrl)")

# SVD dicts for alignment targets
svd_z_dict = gene2emb_pert_svd   # only genes in gene_columns have svd z
svd_u_dict = gene2emb_out_svd

# Build maps for Z (perts)
map_genept_z = None
map_doro_z   = None
map_strseq_z = None
map_strnet_z = None

if ALIGN_EXTERNAL:
    if USE_GENEPT and (GENEPT_WHERE in ["z", "both"]) and (genept_red_z is not None):
        map_genept_z = _fit_ridge_map_dict_to_svd(
            genept_red_z, genept_fb_z, svd_z_dict, genes_needed,
            alpha=ALIGN_RIDGE_ALPHA_Z, min_overlap=ALIGN_MIN_OVERLAP_Z,
            name="GenePT(Z) -> SVD(Z)"
        )

    if USE_DOROTHEA and (DORO_WHERE in ["z", "both"]) and (dorothea_z is not None):
        map_doro_z = _fit_ridge_map_dict_to_svd(
            dorothea_z, dorothea_fb_z, svd_z_dict, genes_needed,
            alpha=ALIGN_RIDGE_ALPHA_Z, min_overlap=ALIGN_MIN_OVERLAP_Z,
            name="DoRothEA(Z) -> SVD(Z)"
        )

    if (USE_STRING_SEQ or USE_STRING_NET) and (STRING_WHERE in ["z", "both"]):
        if USE_STRING_SEQ and (string_seq_z is not None):
            map_strseq_z = _fit_ridge_map_dict_to_svd(
                string_seq_z, string_seq_fb_z, svd_z_dict, genes_needed,
                alpha=ALIGN_RIDGE_ALPHA_Z, min_overlap=ALIGN_MIN_OVERLAP_Z,
                name="STRING_seq(Z) -> SVD(Z)"
            )
        if USE_STRING_NET and (string_net_z is not None):
            map_strnet_z = _fit_ridge_map_dict_to_svd(
                string_net_z, string_net_fb_z, svd_z_dict, genes_needed,
                alpha=ALIGN_RIDGE_ALPHA_Z, min_overlap=ALIGN_MIN_OVERLAP_Z,
                name="STRING_net(Z) -> SVD(Z)"
            )

# Build maps for U (output genes)
map_genept_u = None
map_doro_u   = None
map_strseq_u = None
map_strnet_u = None

if ALIGN_EXTERNAL:
    if USE_GENEPT and (GENEPT_WHERE in ["u", "both"]) and (genept_red_u is not None):
        map_genept_u = _fit_ridge_map_dict_to_svd(
            genept_red_u, genept_fb_u, svd_u_dict, gene_columns,
            alpha=ALIGN_RIDGE_ALPHA_U, min_overlap=ALIGN_MIN_OVERLAP_U,
            name="GenePT(U) -> SVD(U)"
        )

    if USE_DOROTHEA and (DORO_WHERE in ["u", "both"]) and (dorothea_u is not None):
        map_doro_u = _fit_ridge_map_dict_to_svd(
            dorothea_u, dorothea_fb_u, svd_u_dict, gene_columns,
            alpha=ALIGN_RIDGE_ALPHA_U, min_overlap=ALIGN_MIN_OVERLAP_U,
            name="DoRothEA(U) -> SVD(U)"
        )

    if (USE_STRING_SEQ or USE_STRING_NET) and (STRING_WHERE in ["u", "both"]):
        if USE_STRING_SEQ and (string_seq_u is not None):
            map_strseq_u = _fit_ridge_map_dict_to_svd(
                string_seq_u, string_seq_fb_u, svd_u_dict, gene_columns,
                alpha=ALIGN_RIDGE_ALPHA_U, min_overlap=ALIGN_MIN_OVERLAP_U,
                name="STRING_seq(U) -> SVD(U)"
            )
        if USE_STRING_NET and (string_net_u is not None):
            map_strnet_u = _fit_ridge_map_dict_to_svd(
                string_net_u, string_net_fb_u, svd_u_dict, gene_columns,
                alpha=ALIGN_RIDGE_ALPHA_U, min_overlap=ALIGN_MIN_OVERLAP_U,
                name="STRING_net(U) -> SVD(U)"
            )

def _apply_u_aligned(U_out, ext_dict, ext_fb, map_pack, lam, name):
    if (map_pack is None) or (ext_dict is None and ext_fb is None) or (lam == 0):
        return U_out
    add = np.vstack([_map_vec_dict(str(g).upper(), ext_dict, ext_fb, map_pack) for g in gene_columns]).astype(np.float32)
    U_out = U_out + float(lam) * add
    print(f"[align] U_out += {float(lam):.3f} * aligned({name})")
    return U_out

if ALIGN_EXTERNAL:
    if USE_GENEPT and (GENEPT_WHERE in ["u", "both"]):
        U_out = _apply_u_aligned(U_out, genept_red_u, genept_fb_u, map_genept_u, LAM_GENEPT_U, "GenePT(U)")
    if USE_DOROTHEA and (DORO_WHERE in ["u", "both"]):
        U_out = _apply_u_aligned(U_out, dorothea_u, dorothea_fb_u, map_doro_u, LAM_DORO_U, "DoRothEA(U)")
    if (USE_STRING_SEQ or USE_STRING_NET) and (STRING_WHERE in ["u", "both"]):
        if USE_STRING_SEQ:
            U_out = _apply_u_aligned(U_out, string_seq_u, string_seq_fb_u, map_strseq_u, LAM_STRSEQ_U, "STRING_seq(U)")
        if USE_STRING_NET:
            U_out = _apply_u_aligned(U_out, string_net_u, string_net_fb_u, map_strnet_u, LAM_STRNET_U, "STRING_net(U)")

print("[align] U_out final:", U_out.shape, "expected (G, d_out) =", (len(gene_columns), d_out))

def emb_pert(g: str) -> np.ndarray:
    gU = str(g).upper()

    # Base: SVD if available; else h5ad missing fill; else fallback
    if gU in gene2emb_pert_svd:
        z = gene2emb_pert_svd[gU].copy()
    elif gU in missing_pert_h5ad:
        z = missing_pert_h5ad[gU].copy()
    else:
        z = emb_fallback_pert.copy()

    if USE_ZCTRL_ALL and (gU in zctrl_all):
        z = (1.0 - float(ZCTRL_BETA)) * z + float(ZCTRL_BETA) * zctrl_all[gU]

    if ALIGN_EXTERNAL:
        # Add aligned priors in SVD(Z) space
        if USE_GENEPT and (GENEPT_WHERE in ["z", "both"]) and (map_genept_z is not None) and (LAM_GENEPT_Z != 0):
            add = _map_vec_dict(gU, genept_red_z, genept_fb_z, map_genept_z)
            if add is not None:
                z = z + float(LAM_GENEPT_Z) * add

        if USE_DOROTHEA and (DORO_WHERE in ["z", "both"]) and (map_doro_z is not None) and (LAM_DORO_Z != 0):
            add = _map_vec_dict(gU, dorothea_z, dorothea_fb_z, map_doro_z)
            if add is not None:
                z = z + float(LAM_DORO_Z) * add

        if (USE_STRING_SEQ or USE_STRING_NET) and (STRING_WHERE in ["z", "both"]):
            if USE_STRING_SEQ and (map_strseq_z is not None) and (LAM_STRSEQ_Z != 0):
                add = _map_vec_dict(gU, string_seq_z, string_seq_fb_z, map_strseq_z)
                if add is not None:
                    z = z + float(LAM_STRSEQ_Z) * add
            if USE_STRING_NET and (map_strnet_z is not None) and (LAM_STRNET_Z != 0):
                add = _map_vec_dict(gU, string_net_z, string_net_fb_z, map_strnet_z)
                if add is not None:
                    z = z + float(LAM_STRNET_Z) * add

    return z.astype(np.float32)

# Pert embeddings for the 80 training perts
Z_train = np.vstack([emb_pert(g) for g in train_genes]).astype(np.float32)

print("EXP_NAME:", EXP_NAME)
print("U_out:", U_out.shape, "Z_train:", Z_train.shape)

SVD k: 80 gene_emb_all: (5127, 80)
Effective dims: d_pert= 80 d_out= 80
train perts not in gene_columns: 8. Example: ['BRD4', 'CHD4', 'DNAJA3', 'INO80', 'KAT8', 'KDM4A', 'PMEL', 'SETD1A']
val perts not in gene_columns: 8. Example: ['SMARCB1', 'PSMA1', 'CUL1', 'FLT4', 'FOXH1', 'HK2', 'TRAM2', 'DPH2']
[h5ad] cache built: n_cells= 17882 n_genes= 19226 n_ctrl= 1026
[h5ad] embedded missing perts: 16 of 16
[h5ad] embedded ALL perts for blending: 140 of 140
[h5ad] U_ctrl: (5127, 80)
[genept] reduced: z_avail= 138 u_avail= 0 dim_z= 64 dim_u= None
[dorothea] built UniProt->symbol map from STRING: 474839
[dorothea] parsed cols: ['source', 'target', 'is_directed', 'is_stimulation', 'is_inhibition', 'consensus_direction', 'consensus_stimulation', 'consensus_inhibition']
[dorothea] edges loaded: 15267 kept(in-gene-set): 4876
[dorothea] detected UniProt-like IDs: True | mapping size: 474839
[dorothea] Z policy (train perts): row=5 col=28 fb=47 (n=80)
[dorothea] Z policy (genes_needed): row=13 col=42

In [51]:
def _asU_list(xs):
    return [str(x).upper() for x in xs]

genes_neededU  = _asU_list(genes_needed)
gene_columnsU  = _asU_list(gene_columns)
train_genesU   = _asU_list(train_genes)
val_targetsU   = _asU_list(val_targets)

def coverage(name, keysU):
    d = globals().get(name, None)
    if d is None:
        return None
    have = sum([1 for g in keysU if g in d])
    total = len(keysU)
    missing = total - have
    miss_pct = 100.0 * missing / max(1, total)
    miss_list = [g for g in keysU if g not in d]
    return have, total, missing, miss_pct, miss_list

def print_cov_row(src, table, keysU):
    out = coverage(src, keysU)
    if out is None:
        print(f"{src:16s} | {table:18s} | (None)")
        return
    have, total, missing, miss_pct, miss_list = out
    print(f"{src:16s} | {table:18s} | have={have:4d}/{total:<4d}  missing={missing:4d}  ({miss_pct:6.2f}%)")
    if missing > 0:
        print("  - example missing:", miss_list[:12])

print("\n===== Coverage: Z tables (genes_needed / train / val) =====")
for src in ["genept_red_z", "dorothea_z", "string_seq_z", "string_net_z"]:
    print_cov_row(src, "genes_needed (Z)", genes_neededU)
    print_cov_row(src, "train_genes (Z)",  train_genesU)
    print_cov_row(src, "val_targets (Z)",  val_targetsU)
    print("")

print("===== Coverage: U tables (gene_columns) =====")
for src in ["genept_red_u", "dorothea_u", "string_seq_u", "string_net_u"]:
    print_cov_row(src, "gene_columns (U)", gene_columnsU)
    print("")


===== Coverage: Z tables (genes_needed / train / val) =====
genept_red_z     | genes_needed (Z)   | have= 140/140   missing=   0  (  0.00%)
genept_red_z     | train_genes (Z)    | have=  80/80    missing=   0  (  0.00%)
genept_red_z     | val_targets (Z)    | have=  60/60    missing=   0  (  0.00%)

dorothea_z       | genes_needed (Z)   | have= 140/140   missing=   0  (  0.00%)
dorothea_z       | train_genes (Z)    | have=  80/80    missing=   0  (  0.00%)
dorothea_z       | val_targets (Z)    | have=  60/60    missing=   0  (  0.00%)

string_seq_z     | genes_needed (Z)   | have= 140/140   missing=   0  (  0.00%)
string_seq_z     | train_genes (Z)    | have=  80/80    missing=   0  (  0.00%)
string_seq_z     | val_targets (Z)    | have=  60/60    missing=   0  (  0.00%)

string_net_z     | genes_needed (Z)   | (None)
string_net_z     | train_genes (Z)    | (None)
string_net_z     | val_targets (Z)    | (None)

===== Coverage: U tables (gene_columns) =====
genept_red_u     | gene_colu

In [ ]:
def gate_smoothstep(x, a = GATE_A, b = GATE_B):
    t = (x - a) / (b - a)
    t = torch.clamp(t, 0.0, 1.0)
    return t * t * (3.0 - 2.0 * t)

def per_row_weighted_l1_like(delta_true: torch.Tensor, delta_pred: torch.Tensor, eps: float = EPS) -> torch.Tensor:
    """
    Same logic as weighted_l1_like, but returns (N,) per-row
    """
    w = gate_smoothstep(torch.abs(delta_true), a=GATE_A, b=GATE_B)  # (N, G)
    err = torch.abs(delta_pred - delta_true)                        # (N, G)
    num = torch.sum(w * err, dim=1)                                 # (N,)
    den = torch.clamp(torch.sum(w, dim=1), min=eps)                 # (N,)
    return num / den                                                # (N,)

import torch

def weighted_l1_like_rowweighted(
    delta_true: torch.Tensor,     # (N, G)
    delta_pred: torch.Tensor,     # (N, G)
    baseline_wmae: torch.Tensor,  # (N,)
    *,
    eps: float = 1e-8,
    mode: str = "inv_sqrt",       # "inv", "inv_sqrt", "inv_log"
    clamp_min: float = 0.5,
    clamp_max: float = 3.0,
) -> torch.Tensor:
    """
    Weight idea:
      - baseline_wmae small => ratio metric is unforgiving => upweight that row
      - baseline_wmae large => easier => downweight a bit

    Returns: scalar loss
    """
    per_row = per_row_weighted_l1_like(delta_true, delta_pred, eps=eps)  # (N,)

    b = baseline_wmae.to(delta_true.device).to(delta_true.dtype)

    if mode == "inv":
        w = 1.0 / (b + eps)
    elif mode == "inv_sqrt":
        w = 1.0 / torch.sqrt(b + eps)
    elif mode == "inv_log":
        w = 1.0 / torch.log1p(b + eps)

    # clamp to prevent a few rows from dominating training
    w = torch.clamp(w, min=clamp_min, max=clamp_max)

    # normalized weighted mean (stable)
    return torch.sum(w * per_row) / torch.clamp(torch.sum(w), min=eps)

def weighted_cosine_per_row_torch(dt: torch.Tensor, dp: torch.Tensor, w: torch.Tensor, eps: float = 1e-12) -> torch.Tensor:
    # dt, dp, w: (B, G)
    wa = w * dt
    wb = w * dp
    num = torch.sum(wa * wb, dim=1)
    da  = torch.sqrt(torch.sum(wa * wa, dim=1))
    db  = torch.sqrt(torch.sum(wb * wb, dim=1))
    denom = torch.clamp(da * db, min=eps)
    return num / denom  # (B,)


In [53]:
class BilinearDeltaModel(nn.Module):
    def __init__(self, d_pert, d_out, rank_r, dropout):
        super().__init__()
        self.rank_r = rank_r

        self.proj_p = nn.Sequential(
            nn.Linear(d_pert, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )
        self.proj_o = nn.Sequential(
            nn.Linear(d_out, rank_r),
            nn.GELU(),
            nn.Dropout(dropout),
        )

        # biases
        self.bias_global = nn.Parameter(torch.zeros(1))
        self.bias_gene = None  # set via set_gene_bias

    def set_gene_bias(self, G):
        dev = next(self.parameters()).device
        self.bias_gene = torch.nn.Parameter(torch.zeros(G, device=dev))
        self.bias_global = torch.nn.Parameter(torch.zeros(1, device=dev))

    def forward(self, z_pert, u_out):
        p = self.proj_p(z_pert)    # (B, R)
        o = self.proj_o(u_out)     # (G, R)
        y = p @ o.T                # (B, G)
        y = y + self.bias_gene[None, :] + self.bias_global
        return y

In [54]:
Y = D_train.astype(np.float32)
G = Y.shape[1]
N = Y.shape[0]

Uo_t = torch.tensor(U_out, device=device) # (G, d_out)
Zt = torch.tensor(Z_train, device=device) # (N, d_pert)
Yt = torch.tensor(Y, device=device) # (N, G)

print("N:", N, "G:", G, "device:", device)

N: 80 G: 5127 device: cuda


In [55]:
gt_df = pd.read_csv("Data/training_data_ground_truth_table.csv")
sol_aligned = gt_df.set_index("pert_id").loc[train_genes].reset_index()

# Keep only needed columns for speed: pert_id + genes + weights + baseline
w_cols = [f"w_{g}" for g in gene_columns]
sol_aligned = sol_aligned[["pert_id"] + list(gene_columns) + w_cols + ["baseline_wmae"]]
baseline_wmae = sol_aligned["baseline_wmae"].to_numpy(np.float32)
baseline_wmae_t = torch.tensor(baseline_wmae, device=device, dtype=torch.float32)

W_gene = sol_aligned[w_cols].to_numpy(np.float32)   # (N, G)

W64 = W_gene.astype(np.float64)
row_sums = W64.sum(axis=1, keepdims=True)
W64 *= (G / np.maximum(row_sums, 1e-300))
W64[:, -1] += (G - W64.sum(axis=1))
W_gene = W64.astype(np.float32)

In [56]:
from myllia_metric import score as kaggle_score

def score_delta(dt, dp, idx):
    dt = np.asarray(dt, np.float64)
    dp = np.asarray(dp, np.float64)
    w  = W_gene[idx].astype(np.float64, copy=False)
    base = baseline_wmae[idx].astype(np.float64, copy=False)

    abs_err = np.abs(dt - dp)
    pred_wmae = np.mean(abs_err * w, axis=1)
    pred_wmae = np.maximum(pred_wmae, 1e-12)
    base = np.maximum(base, 1e-12)

    terms = np.log2(base / pred_wmae)
    terms = np.minimum(terms, 5.0)
    sum_wmae = float(np.sum(terms))
    mean_term = float(np.mean(terms))

    a = dp.ravel()
    b = dt.ravel()
    x = np.maximum(np.abs(a), np.abs(b))
    t = np.clip(x / 0.2, 0.0, 1.0)
    w_gate = t * t * (3.0 - 2.0 * t)
    w2 = w_gate * w_gate

    num = np.sum(w2 * a * b)
    den = np.sqrt(np.sum(w2 * a * a)) * np.sqrt(np.sum(w2 * b * b))
    wcos = 0.0 if den < 1e-12 else float(num / den)
    wcos_pos = max(0.0, wcos)

    raw = sum_wmae * wcos_pos
    score = round(raw, 5)

    # normalized (scale-free) metric
    score_per_row = mean_term * wcos_pos

    return {
        "score": score,
        "raw": float(raw),
        "sum_wmae": sum_wmae,
        "mean_term": mean_term,
        "wcos": wcos,
        "score_per_row": float(score_per_row),
        "n_rows": int(len(idx)),
    }

In [57]:
# =============================
# EXTRA H5AD DATA: BOOTSTRAP DELTAS (precompute once)
# =============================
boot_raw = None
h5_mean_raw = None
boot_ok = None

if AUGMENT_H5AD:
    cache = load_h5ad_ctrl_cache(H5AD_PATH, gene_columns)

    Xn = cache["Xn"]               # sparse (n_cells, 19226) normalized CPM10K + log2
    out_idx = cache["out_idx"]     # (5127,)
    obs_pertU = cache["obs_pertU"] # (n_cells,) uppercase pert labels
    ctrl_mask = cache["ctrl_mask"]

    # control mean over output genes (in same h5ad normalized space)
    ctrl_mean = Xn[ctrl_mask][:, out_idx].mean(axis=0)
    if sparse.issparse(ctrl_mean):
        ctrl_mean = ctrl_mean.A
    ctrl_mean = np.asarray(ctrl_mean).ravel().astype(np.float32)  # (5127,)

    N = len(train_genes)
    G = len(gene_columns)

    h5_mean_raw = np.zeros((N, G), dtype=np.float32)
    boot_raw = np.zeros((N, BOOT_K, G), dtype=np.float32)
    boot_ok = np.zeros((N,), dtype=np.int32)

    rng = np.random.RandomState(BOOT_SEED)

    for i, g in enumerate(train_genes.tolist()):
        gU = str(g).upper()
        rows = np.where(obs_pertU == gU)[0]
        if len(rows) == 0:
            continue

        boot_ok[i] = 1

        # mean delta using ALL cells for this pert
        Xi = Xn[rows][:, out_idx].mean(axis=0)
        if sparse.issparse(Xi):
            Xi = Xi.A
        Xi = np.asarray(Xi).ravel().astype(np.float32)
        h5_mean_raw[i] = (Xi - ctrl_mean)

        # bootstraps
        for k in range(BOOT_K):
            samp = rng.choice(rows, size=min(int(BOOT_M), len(rows)), replace=True)
            Xk = Xn[samp][:, out_idx].mean(axis=0)
            if sparse.issparse(Xk):
                Xk = Xk.A
            Xk = np.asarray(Xk).ravel().astype(np.float32)
            boot_raw[i, k] = (Xk - ctrl_mean)

    print("[aug] boot_ok:", int(boot_ok.sum()), "/", N)
    print("[aug] h5_mean_raw:", h5_mean_raw.shape, "boot_raw:", boot_raw.shape)

[aug] boot_ok: 80 / 80
[aug] h5_mean_raw: (80, 5127) boot_raw: (80, 32, 5127)


In [ ]:
ALPHA_GRID = np.linspace(0.5, 1.8, 100)
def apply_shrink(pred, baseline, alpha):
    # pred: (B,G) ; baseline: (G,)
    return float(alpha) * pred + (1.0 - float(alpha)) * baseline[None, :]

def train_one_fold(tr_idx, va_idx, seed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)

    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    tr_idx = np.asarray(tr_idx)
    va_idx = np.asarray(va_idx)
    va_idx_t = torch.tensor(va_idx, device=device, dtype=torch.long)

    aug_enabled = (AUGMENT_H5AD and (boot_raw is not None) and (h5_mean_raw is not None) and (boot_ok is not None))

    slopes_t = None
    intercepts_t = None
    rng_aug = None

    if aug_enabled:
        ok_mask = (boot_ok[tr_idx] == 1)
        tr_fit = tr_idx[ok_mask]

        if len(tr_fit) >= AUG_MIN_FIT_PERTS:
            A = h5_mean_raw[tr_fit].astype(np.float32)  # (n_fit, G) in h5ad-space
            B = Y[tr_fit].astype(np.float32)            # (n_fit, G) in means-space

            # per-gene affine: B ~= slope * A + intercept
            Am = A.mean(axis=0)
            Bm = B.mean(axis=0)
            Av = ((A - Am[None, :]) ** 2).mean(axis=0) + 1e-6
            Cov = ((A - Am[None, :]) * (B - Bm[None, :])).mean(axis=0)

            slopes = (Cov / Av).astype(np.float32)
            slopes = np.clip(slopes, float(AUG_SLOPE_CLAMP_MIN), float(AUG_SLOPE_CLAMP_MAX)).astype(np.float32)
            intercepts = (Bm - slopes * Am).astype(np.float32)

            slopes_t = torch.tensor(slopes[None, :], device=device, dtype=torch.float32)       # (1, G)
            intercepts_t = torch.tensor(intercepts[None, :], device=device, dtype=torch.float32) # (1, G)

            rng_aug = np.random.RandomState(seed + 2027)
            #print("[aug] enabled: fit_perts=", len(tr_fit), "slope_mean=", float(slopes.mean()), "slope_std=", float(slopes.std()))
        else:
            aug_enabled = False
            print("[aug] disabled: not enough fit perts in this fold:", len(tr_fit))

    best_score = -1e18
    best_alpha = 0.0
    best_epoch = 0
    best_state = None
    best_va_pred = None  # unshrunk predictions at best checkpoint
    patience = 0

    for epoch in range(1, EPOCHS + 1):
        model.train()
        perm = tr_idx.copy()
        np.random.shuffle(perm)

        for start in range(0, len(perm), BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)

            if aug_enabled and (slopes_t is not None):
                kk = rng_aug.randint(0, BOOT_K, size=len(b))
                dt_raw_np = boot_raw[b, kk, :]  # (B, G) in h5ad-space
                dt_raw_t = torch.tensor(dt_raw_np, device=device, dtype=torch.float32)

                # map to means-space using train-only affine
                dt_aug = dt_raw_t * slopes_t + intercepts_t  # (B, G)

                # mix with original means target for stability
                dt_b = (1.0 - float(AUG_P)) * Yt.index_select(0, b_t) + float(AUG_P) * dt_aug
            else:
                dt_b = Yt.index_select(0, b_t)               # (B, G)

            bw_b = baseline_wmae_t.index_select(0, b_t)     # (B,)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        if epoch % EVAL_EVERY == 0 or epoch == EPOCHS:
            model.eval()
            with torch.no_grad():
                va_pred = model(Zt.index_select(0, va_idx_t), Uo_t).detach().cpu().numpy().astype(np.float32)

            va_true = Y[va_idx]

            # alpha sweep (0..0.7) on this fold
            sc_best = -1e18
            a_best = 0.0
            for a in ALPHA_GRID:
                pred_a = apply_shrink(va_pred, delta_baseline, float(a))
                sc = score_delta(va_true, pred_a, va_idx)["score"]
                if sc > sc_best:
                    sc_best = sc
                    a_best = float(a)

            if sc_best > best_score:
                best_score = sc_best
                best_alpha = a_best
                best_epoch = epoch
                best_state = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
                best_va_pred = va_pred.copy()
                patience = 0
            else:
                patience += 1

            if patience >= PATIENCE:
                break

    return best_score, best_alpha, best_epoch, best_state, best_va_pred

# --- CV: collect OOF preds and optimize a GLOBAL alpha on OOF only ---
kf = KFold(n_splits=8, shuffle=True, random_state=SEED)

oof_pred = np.zeros_like(Y, dtype=np.float32)
oof_hit  = np.zeros((N,), dtype=np.int32)

fold_scores = []          # mean score per fold across seeds
fold_scores_seeds = []
fold_epochs = []          # median/mean epoch across seeds per fold (pick one)
fold_alphas = []          # mean alpha across seeds per fold (just for logging)

for fold, (tr_idx, va_idx) in enumerate(kf.split(np.arange(N)), 1):

    # Collect per-seed results for this fold
    seed_scores = []
    seed_alphas = []
    seed_epochs = []
    seed_va_preds = []

    for s in MODEL_SEEDS:
        best_score, best_alpha, best_epoch, best_state, best_va_pred = train_one_fold(tr_idx, va_idx, seed=int(s))

        seed_scores.append(float(best_score))
        seed_alphas.append(float(best_alpha))
        seed_epochs.append(int(best_epoch))
        seed_va_preds.append(best_va_pred.astype(np.float32, copy=False))

    fold_score_mean = float(np.mean(seed_scores))
    fold_scores_seeds.append(seed_scores)

    # Logging-only summaries
    fold_alphas.append(float(np.mean(seed_alphas)))
    fold_epochs.append(int(np.median(seed_epochs)))

    va_pred_mean = np.mean(np.stack(seed_va_preds, axis=0), axis=0).astype(np.float32)  # (B, G)

    oof_pred[va_idx] = va_pred_mean
    oof_hit[va_idx] += 1
    fold_factor = float(N / len(va_idx))  # ~8.0 for 8-fold CV
    seeds_str = ", ".join([f"{sc * fold_factor:.6f}" for sc in seed_scores])
    fold_score_scaled = fold_score_mean * fold_factor
    fold_scores.append(fold_score_scaled)
    print(
        f"fold {fold}: score_mean={fold_score_scaled:.6f} "
        f"scores=[{seeds_str}] "
        f"alpha_mean={np.mean(seed_alphas):.3f} "
        f"epoch_median={int(np.median(seed_epochs))}"
    )

if not np.all(oof_hit == 1):
    print("[warn] OOF coverage not 1 everywhere. min/max:", int(oof_hit.min()), int(oof_hit.max()))

print("cv mean:", float(np.mean(fold_scores)), "std:", float(np.std(fold_scores)))

EPOCHS_MED = int(np.median(fold_epochs))
print("median best_epoch =", EPOCHS_MED)

best_global_alpha = 0.0
best_global_score = -1e18

all_idx = np.arange(N, dtype=np.int64)

for a in ALPHA_GRID:
    pred_a = apply_shrink(oof_pred, delta_baseline, float(a))
    sc = score_delta(Y, pred_a, all_idx)["score"]
    if sc > best_global_score:
        best_global_score = sc
        best_global_alpha = float(a)

print("OOF global alpha:", best_global_alpha, "OOF score:", best_global_score)
ALPHA_SHRINK = best_global_alpha

fold 1: score_mean=6.435920 scores=[6.226080, 6.420560, 6.661120] alpha_mean=1.003 epoch_median=75
fold 2: score_mean=4.853867 scores=[4.854880, 4.772480, 4.934240] alpha_mean=1.021 epoch_median=70
fold 3: score_mean=4.580213 scores=[4.541680, 4.720720, 4.478240] alpha_mean=0.806 epoch_median=95
fold 4: score_mean=4.501573 scores=[4.370560, 4.628080, 4.506080] alpha_mean=1.021 epoch_median=80
fold 5: score_mean=4.737093 scores=[4.742720, 4.740400, 4.728160] alpha_mean=0.977 epoch_median=80
fold 6: score_mean=7.225653 scores=[7.352400, 7.085760, 7.238800] alpha_mean=0.898 epoch_median=80
fold 7: score_mean=7.514187 scores=[7.666320, 7.482320, 7.393920] alpha_mean=0.872 epoch_median=50
fold 8: score_mean=4.522320 scores=[4.575600, 4.490080, 4.501280] alpha_mean=0.859 epoch_median=50
cv mean: 5.546353333333333 std: 1.2089158841246521
median best_epoch = 77
OOF global alpha: 0.9070707070707071 OOF score: 5.26086


|Type|cv mean|OOF score|
|-------|-----------|-------------|
|Dor Gen All|5.74874|5.68997|
|No Dorothea |5.56606| 5.42451 |
|No Genept|5.536723|5.3467|
| Just Dorothea|5.44974|5.23232|
|Baseline| 5.417313| 5.18041|
|New Baseline|5.35713|5.18589|
|*ALL New*|*5.714639*|*5.57851*|

fold 1: score_mean=6.399093 scores=[6.195440, 6.369680, 6.632160] alpha_mean=0.999 epoch_median=70
fold 2: score_mean=4.844693 scores=[4.841360, 4.766080, 4.926640] alpha_mean=1.021 epoch_median=70
fold 3: score_mean=4.587520 scores=[4.547680, 4.727440, 4.487440] alpha_mean=0.806 epoch_median=95
fold 4: score_mean=4.490960 scores=[4.364800, 4.624320, 4.483760] alpha_mean=1.021 epoch_median=80
fold 5: score_mean=4.737307 scores=[4.748880, 4.731760, 4.731280] alpha_mean=0.977 epoch_median=80
fold 6: score_mean=7.216587 scores=[7.340000, 7.076080, 7.233680] alpha_mean=0.894 epoch_median=80
fold 7: score_mean=7.498400 scores=[7.656560, 7.465600, 7.373040] alpha_mean=0.868 epoch_median=50
fold 8: score_mean=4.492933 scores=[4.547600, 4.463040, 4.468160] alpha_mean=0.850 epoch_median=50
cv mean: 5.533436666666667 std: 1.2049787224262511
median best_epoch = 75
OOF global alpha: 0.9070707070707071 OOF score: 5.25067 * INCREASE LB

In [ ]:
def fit_full_model(seed, epochs_fixed):
    np.random.seed(seed)
    torch.manual_seed(seed)

    model = BilinearDeltaModel(
        d_pert=Zt.shape[1],
        d_out=Uo_t.shape[1],
        rank_r=RANK_R,
        dropout=DROPOUT
    ).to(device)

    model.set_gene_bias(G)
    opt = torch.optim.AdamW(model.parameters(), lr=LR, weight_decay=WD)

    aug_enabled = False
    slopes_t = None
    intercepts_t = None
    rng_aug = None

    if ("AUGMENT_H5AD" in globals()) and AUGMENT_H5AD:
        assert (boot_raw is not None) and (h5_mean_raw is not None) and (boot_ok is not None), \
            "AUGMENT_H5AD=True but boot_raw/h5_mean_raw/boot_ok not built. Run the bootstrap precompute cell."

        fit_idx = np.where(boot_ok == 1)[0]
        if len(fit_idx) >= AUG_MIN_FIT_PERTS:
            A = h5_mean_raw[fit_idx].astype(np.float32)  # (n_fit, G) h5ad-space
            B = Y[fit_idx].astype(np.float32)            # (n_fit, G) means-space

            Am = A.mean(axis=0)
            Bm = B.mean(axis=0)
            Av = ((A - Am[None, :]) ** 2).mean(axis=0) + 1e-6
            Cov = ((A - Am[None, :]) * (B - Bm[None, :])).mean(axis=0)

            slopes = (Cov / Av).astype(np.float32)
            slopes = np.clip(slopes, float(AUG_SLOPE_CLAMP_MIN), float(AUG_SLOPE_CLAMP_MAX)).astype(np.float32)
            intercepts = (Bm - slopes * Am).astype(np.float32)

            slopes_t = torch.tensor(slopes[None, :], device=device, dtype=torch.float32)          # (1, G)
            intercepts_t = torch.tensor(intercepts[None, :], device=device, dtype=torch.float32)  # (1, G)

            rng_aug = np.random.RandomState(seed + 2027)
            aug_enabled = True
            print(f"[aug/full] enabled: fit_perts={len(fit_idx)} slope_mean={float(slopes.mean()):.6f} slope_std={float(slopes.std()):.6f}")
        else:
            print(f"[aug/full] disabled: not enough fit perts with h5ad coverage ({len(fit_idx)})")

    all_idx = np.arange(N, dtype=np.int64)

    for epoch in range(1, int(epochs_fixed) + 1):
        model.train()
        perm = np.arange(N)
        np.random.shuffle(perm)

        for start in range(0, N, BATCH_GENES):
            b = perm[start:start + BATCH_GENES]
            b_t = torch.tensor(b, device=device, dtype=torch.long)

            pred = model(Zt.index_select(0, b_t), Uo_t)

            if aug_enabled and (slopes_t is not None):
                kk = rng_aug.randint(0, BOOT_K, size=len(b))
                dt_raw_np = boot_raw[b, kk, :]  # (B, G) h5ad-space
                dt_raw_t = torch.tensor(dt_raw_np, device=device, dtype=torch.float32)

                dt_aug = dt_raw_t * slopes_t + intercepts_t  # (B, G) means-space
                dt_b = (1.0 - float(AUG_P)) * Yt.index_select(0, b_t) + float(AUG_P) * dt_aug
            else:
                dt_b = Yt.index_select(0, b_t)

            bw_b = baseline_wmae_t.index_select(0, b_t)

            loss = weighted_l1_like_rowweighted(
                dt_b, pred, bw_b,
                mode="inv_sqrt",
                clamp_min=0.5,
                clamp_max=3.0,
            )

            opt.zero_grad()
            loss.backward()
            torch.nn.utils.clip_grad_norm_(model.parameters(), GRAD_CLIP)
            opt.step()

        # Monitor score in submission space (means-space + shrink)
        if (epoch % EVAL_EVERY == 0) or (epoch == int(epochs_fixed)):
            model.eval()
            with torch.no_grad():
                pred_np = model(Zt, Uo_t).detach().cpu().numpy().astype(np.float32)

            pred_np = apply_shrink(pred_np, delta_baseline, float(ALPHA_SHRINK))

            # Score_delta_fast expects idx
            s = score_delta(Y, pred_np, all_idx)

            if "wcos" in s:
                print(f"[seed {seed}] epoch={epoch:4d} train_score={s['score']:.6f} wcos={s['wcos']:.6f} mean_term={s.get('mean_term', float('nan')):.6f} alpha={float(ALPHA_SHRINK):.3f}")
            else:
                print(f"[seed {seed}] epoch={epoch:4d} train_score={s['score']:.6f} alpha={float(ALPHA_SHRINK):.3f}")

    model.eval()
    return model


# Refit ensemble on ALL perts, using CV-calibrated epoch + OOF-calibrated alpha
models = [fit_full_model(int(sd), epochs_fixed=int(EPOCHS_MED)) for sd in MODEL_SEEDS]
print("Refit models:", len(models))


def predict_delta_gene(gene_symbol: str) -> np.ndarray:
    z = torch.tensor(emb_pert(gene_symbol)[None, :].astype(np.float32), device=device)
    preds = []
    with torch.no_grad():
        for m in models:
            y = m(z, Uo_t).detach().cpu().numpy().astype(np.float32)[0]
            preds.append(y)
    yhat = np.mean(np.stack(preds, axis=0), axis=0).astype(np.float32)
    yhat = apply_shrink(yhat[None, :], delta_baseline, float(ALPHA_SHRINK))[0].astype(np.float32)
    return yhat


# Build submission from sample_submission.csv
sub = df_sub.copy()
sub["pert_id"] = sub["pert_id"].astype(str)
sub_gene_cols = [c for c in sub.columns if c != "pert_id"]

# ensure order matches gene_columns
idx_map = {str(g).upper(): i for i, g in enumerate(gene_columns)}
perm = [idx_map[str(g).upper()] for g in sub_gene_cols]

# fill default baseline for unknown test perts
sub.loc[:, sub_gene_cols] = np.tile(delta_baseline[perm][None, :], (len(sub), 1))

# fill known val perts (pert_1..pert_60)
hit = 0
for pid, gene in val_map.items():
    vec = predict_delta_gene(gene)[perm]
    m = (sub["pert_id"] == str(pid))
    if m.any():
        sub.loc[m, sub_gene_cols] = vec[None, :]
        hit += int(m.sum())

#out_path = "optimized_dorothea_aligned.csv"
#sub.to_csv(out_path, index=False)
#print("[ok] wrote:", out_path, "| filled:", hit)

[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 90] epoch=   5 train_score=-1.975010 wcos=0.281232 mean_term=-0.087784 alpha=0.907
[seed 90] epoch=  10 train_score=0.281340 wcos=0.367928 mean_term=0.009558 alpha=0.907
[seed 90] epoch=  15 train_score=3.085020 wcos=0.387857 mean_term=0.099425 alpha=0.907
[seed 90] epoch=  20 train_score=5.112240 wcos=0.380323 mean_term=0.168023 alpha=0.907
[seed 90] epoch=  25 train_score=6.329560 wcos=0.370180 mean_term=0.213733 alpha=0.907
[seed 90] epoch=  30 train_score=7.017520 wcos=0.362584 mean_term=0.241928 alpha=0.907
[seed 90] epoch=  35 train_score=7.409330 wcos=0.357722 mean_term=0.258906 alpha=0.907
[seed 90] epoch=  40 train_score=7.676010 wcos=0.354673 mean_term=0.270531 alpha=0.907
[seed 90] epoch=  45 train_score=7.867550 wcos=0.353271 mean_term=0.278382 alpha=0.907
[seed 90] epoch=  50 train_score=8.031260 wcos=0.352332 mean_term=0.284932 alpha=0.907
[seed 90] epoch=  55 train_score=8.210130 wcos=0.353699 

[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 90] epoch=   5 train_score=-1.974260 wcos=0.281315 mean_term=-0.087725 alpha=0.907
[seed 90] epoch=  10 train_score=0.283090 wcos=0.367997 mean_term=0.009616 alpha=0.907
[seed 90] epoch=  15 train_score=3.087100 wcos=0.387926 mean_term=0.099475 alpha=0.907
[seed 90] epoch=  20 train_score=5.114370 wcos=0.380418 mean_term=0.168051 alpha=0.907
[seed 90] epoch=  25 train_score=6.331830 wcos=0.370280 mean_term=0.213751 alpha=0.907
[seed 90] epoch=  30 train_score=7.021200 wcos=0.362690 mean_term=0.241984 alpha=0.907
[seed 90] epoch=  35 train_score=7.414330 wcos=0.357842 mean_term=0.258994 alpha=0.907
[seed 90] epoch=  40 train_score=7.682780 wcos=0.354808 mean_term=0.270667 alpha=0.907
[seed 90] epoch=  45 train_score=7.875430 wcos=0.353427 mean_term=0.278538 alpha=0.907
[seed 90] epoch=  50 train_score=8.041160 wcos=0.352518 mean_term=0.285133 alpha=0.907
[seed 90] epoch=  55 train_score=8.221530 wcos=0.353891 mean_term=0.290398 alpha=0.907
[seed 90] epoch=  60 train_score=8.351000 wcos=0.354042 mean_term=0.294845 alpha=0.907
[seed 90] epoch=  65 train_score=8.521800 wcos=0.355255 mean_term=0.299848 alpha=0.907
[seed 90] epoch=  70 train_score=8.668590 wcos=0.356735 mean_term=0.303748 alpha=0.907
[seed 90] epoch=  75 train_score=8.834390 wcos=0.358736 mean_term=0.307831 alpha=0.907
[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 70] epoch=   5 train_score=-2.029990 wcos=0.300449 mean_term=-0.084457 alpha=0.907
[seed 70] epoch=  10 train_score=0.352270 wcos=0.367745 mean_term=0.011974 alpha=0.907
[seed 70] epoch=  15 train_score=3.045170 wcos=0.378815 mean_term=0.100484 alpha=0.907
[seed 70] epoch=  20 train_score=4.966120 wcos=0.369603 mean_term=0.167954 alpha=0.907
[seed 70] epoch=  25 train_score=6.290320 wcos=0.366810 mean_term=0.214359 alpha=0.907
[seed 70] epoch=  30 train_score=6.912580 wcos=0.358877 mean_term=0.240771 alpha=0.907
[seed 70] epoch=  35 train_score=7.291390 wcos=0.353778 mean_term=0.257626 alpha=0.907
[seed 70] epoch=  40 train_score=7.540020 wcos=0.351210 mean_term=0.268358 alpha=0.907
[seed 70] epoch=  45 train_score=7.712860 wcos=0.349994 mean_term=0.275464 alpha=0.907
[seed 70] epoch=  50 train_score=7.841590 wcos=0.349305 mean_term=0.280614 alpha=0.907
[seed 70] epoch=  55 train_score=7.926460 wcos=0.348452 mean_term=0.284345 alpha=0.907
[seed 70] epoch=  60 train_score=8.055660 wcos=0.349633 mean_term=0.288004 alpha=0.907
[seed 70] epoch=  65 train_score=8.156050 wcos=0.350474 mean_term=0.290894 alpha=0.907
[seed 70] epoch=  70 train_score=8.302670 wcos=0.351878 mean_term=0.294941 alpha=0.907
[seed 70] epoch=  75 train_score=8.414150 wcos=0.353184 mean_term=0.297796 alpha=0.907
[aug/full] enabled: fit_perts=80 slope_mean=0.980505 slope_std=0.062581
[seed 80] epoch=   5 train_score=-1.945340 wcos=0.275113 mean_term=-0.088388 alpha=0.907
[seed 80] epoch=  10 train_score=0.330910 wcos=0.373850 mean_term=0.011064 alpha=0.907
[seed 80] epoch=  15 train_score=2.998010 wcos=0.377995 mean_term=0.099142 alpha=0.907
[seed 80] epoch=  20 train_score=5.103330 wcos=0.378063 mean_term=0.168733 alpha=0.907
[seed 80] epoch=  25 train_score=6.282780 wcos=0.366795 mean_term=0.214111 alpha=0.907
[seed 80] epoch=  30 train_score=6.969600 wcos=0.359670 mean_term=0.242222 alpha=0.907
[seed 80] epoch=  35 train_score=7.312050 wcos=0.353766 mean_term=0.258365 alpha=0.907
[seed 80] epoch=  40 train_score=7.640380 wcos=0.353875 mean_term=0.269883 alpha=0.907
[seed 80] epoch=  45 train_score=7.843660 wcos=0.353154 mean_term=0.277629 alpha=0.907
[seed 80] epoch=  50 train_score=7.983690 wcos=0.353358 mean_term=0.282422 alpha=0.907
[seed 80] epoch=  55 train_score=8.120330 wcos=0.352856 mean_term=0.287665 alpha=0.907
[seed 80] epoch=  60 train_score=8.261150 wcos=0.353591 mean_term=0.292045 alpha=0.907
[seed 80] epoch=  65 train_score=8.395990 wcos=0.354853 mean_term=0.295756 alpha=0.907
[seed 80] epoch=  70 train_score=8.539820 wcos=0.355453 mean_term=0.300315 alpha=0.907
[seed 80] epoch=  75 train_score=8.681180 wcos=0.357024 mean_term=0.303943 alpha=0.907
Refit models: 3
[ok] wrote: test_fill.csv | filled: 60